# FusionLoRA Training — Google Colab
Runs two 336px FusionLoRA experiments sequentially.

**Before running:** upload the following to your Google Drive under `My Drive/airbnb_collab/`:
- `data/` folder (all parquets + joblib files)
- `images/processed_336/` folder

Runtime: **GPU** (T4 recommended, 15GB VRAM).

In [ ]:
# ── 1. Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers peft sentencepiece joblib pyarrow lightgbm
!pip install -q torchao>=0.16.0

In [ ]:
# ── 3. Clone repo and symlink data from Drive ────────────────────────────────
import os, shutil

REPO = '/content/airbnb'
DRIVE_BASE = '/content/drive/MyDrive/airbnb_collab'

if not os.path.exists(REPO):
    !git clone https://github.com/asquik/ML-AirBnb-Price-Estimator.git {REPO}
else:
    !cd {REPO} && git pull

os.makedirs(f'{REPO}/images', exist_ok=True)

for link, target in [
    (f'{REPO}/data',                    f'{DRIVE_BASE}/data'),
    (f'{REPO}/images/processed_336',    f'{DRIVE_BASE}/images/processed_336'),
    (f'{REPO}/outputs',                 f'{DRIVE_BASE}/outputs'),
]:
    # Remove whatever is there (real dir or stale symlink)
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    os.symlink(target, link)
    print(f'Linked {link} -> {target}')

os.makedirs(f'{DRIVE_BASE}/outputs/runs', exist_ok=True)

# Verify
import glob
parquets = glob.glob(f'{REPO}/data/*.parquet')
print(f'\nFound {len(parquets)} parquet files')
images = glob.glob(f'{REPO}/images/processed_336/*.jpg')
print(f'Found {len(images)} images in processed_336/')
print('\nSetup complete.')

In [ ]:
# ── 4. Smoke test — verify data loads and GPU is available ──────────────────
import subprocess, sys

cmd = [sys.executable, 'scripts/models/fusion_lora.py',
       '--variant',    'normal_bc',
       '--image-size', '336',
       '--lora-rank',  '16',
       '--fusion-head','deep_256',
       '--smoke-test']

proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('Return code:', proc.returncode)

In [ ]:
# ── 5. Run 1: rank=16, deep_256, 336px ──────────────────────────────────────
import subprocess, sys

cmd = [sys.executable, 'scripts/models/fusion_lora.py',
       '--variant',     'normal_bc',
       '--image-size',  '336',
       '--lora-rank',   '16',
       '--fusion-head', 'deep_256',
       '--lr-head',     '5e-5',
       '--batch-size',  '8',
       '--accum-steps', '4',
       '--run-name',    'colab_rank16_deep256_336px_lr5e5']

proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('Return code:', proc.returncode)

In [ ]:
# ── 6. Run 2: rank=8, deep_256, 336px ───────────────────────────────────────
import subprocess, sys

cmd = [sys.executable, 'scripts/models/fusion_lora.py',
       '--variant',     'normal_bc',
       '--image-size',  '336',
       '--lora-rank',   '8',
       '--fusion-head', 'deep_256',
       '--lr-head',     '5e-5',
       '--batch-size',  '8',
       '--accum-steps', '4',
       '--run-name',    'colab_rank8_deep256_336px_lr5e5']

proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('Return code:', proc.returncode)

In [ ]:
# ── 7. Check results ─────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

log_path = Path(f'{DRIVE_BASE}/outputs/master_runs_log.csv')
if not log_path.exists():
    print("No runs completed yet — master_runs_log.csv not found.")
else:
    log = pd.read_csv(log_path)
    cols = ['run_id', 'model_type', 'dataset_variant', 'image_size', 'lora_rank',
            'fusion_head', 'test_rmse_raw', 'test_r2', 'training_time_minutes']
    lora_runs = log[log['model_type'] == 'FusionLoRA']
    if lora_runs.empty:
        print("No FusionLoRA runs in log yet.")
    else:
        print(lora_runs[cols].to_string(index=False))